<a href="https://colab.research.google.com/github/elsa-paul11/de-portfolio-2026/blob/main/module-02-spark-internals/notebooks/m2_d2_spark_internals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pyspark==4.0.0 -q

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from decimal import Decimal
import random
import logging
import sys

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.1/434.1 MB 3.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [2]:
# Force reconfigure logging
logger = logging.getLogger("module1")
logger.setLevel(logging.INFO)

if not logger.handlers:
    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(logging.Formatter(
        "%(asctime)s | %(levelname)s | %(message)s"
    ))
    logger.addHandler(handler)

logger.propagate = False  # Prevent Colab's root logger from interfering

logger.info("Logger working correctly")

2026-06-03 18:13:10,581 | INFO | Logger working correctly


In [3]:


spark = SparkSession.builder \
    .appName("day2_spark_internals") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.sql.session.timeZone", "UTC") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
logger.info(f"Spark ready | version={spark.version}")

2026-06-03 18:13:34,664 | INFO | Spark ready | version=4.0.0


In [4]:
# Generate orders data
random.seed(42)
statuses = ["COMPLETED", "PENDING", "CANCELLED", "REFUNDED"]
regions  = ["IN-SOUTH", "IN-NORTH", "IN-WEST", "IN-EAST"]

rows = []
for i in range(1, 10_001):
    rows.append((
        i,
        random.randint(1, 5000),
        Decimal(str(round(random.uniform(10.0, 5000.0), 2))),
        random.choice(statuses),
        random.choice(regions),
    ))

schema = StructType([
    StructField("order_id",    IntegerType(),      False),
    StructField("customer_id", IntegerType(),      False),
    StructField("amount",      DecimalType(10,2),  False),
    StructField("status",      StringType(),       True),
    StructField("region",      StringType(),       True),
])

df = spark.createDataFrame(rows, schema=schema)

print(f"Number of Spark partitions: {df.rdd.getNumPartitions()}")

Number of Spark partitions: 2


In [5]:
# Watch how nothing executes until an action is called

import time

# These lines execute INSTANTLY — no data processing
df_filtered  = df.filter(F.col("status") == "COMPLETED")
df_grouped   = df_filtered.groupBy("region").agg(
    F.sum("amount").alias("total_amount"),
    F.count("*").alias("order_count")
)

logger.info("Above lines were instant — no data moved yet")

# NOW data actually processes — this is the action
start = time.time()
df_grouped.show()
duration = time.time() - start

logger.info(f"Action completed in {duration:.2f} seconds")

2026-06-03 18:15:35,649 | INFO | Above lines were instant — no data moved yet
+--------+------------+-----------+
|  region|total_amount|order_count|
+--------+------------+-----------+
| IN-WEST|  1593682.18|        641|
| IN-EAST|  1569083.49|        620|
|IN-NORTH|  1575983.03|        629|
|IN-SOUTH|  1412816.40|        587|
+--------+------------+-----------+

2026-06-03 18:15:40,715 | INFO | Action completed in 5.06 seconds


In [6]:
# This shows you exactly what Spark PLANS to do
# before it actually does it
# Read bottom to top — that is the execution order

print("=== EXECUTION PLAN ===")
df_grouped.explain(mode="formatted")

=== EXECUTION PLAN ===
== Physical Plan ==
AdaptiveSparkPlan (7)
+- HashAggregate (6)
   +- Exchange (5)
      +- HashAggregate (4)
         +- Project (3)
            +- Filter (2)
               +- Scan ExistingRDD (1)


(1) Scan ExistingRDD
Output [5]: [order_id#0, customer_id#1, amount#2, status#3, region#4]
Arguments: [order_id#0, customer_id#1, amount#2, status#3, region#4], MapPartitionsRDD[4] at applySchemaToPythonRDD at NativeMethodAccessorImpl.java:0, ExistingRDD, UnknownPartitioning(0)

(2) Filter
Input [5]: [order_id#0, customer_id#1, amount#2, status#3, region#4]
Condition : (isnotnull(status#3) AND (status#3 = COMPLETED))

(3) Project
Output [2]: [amount#2, region#4]
Input [5]: [order_id#0, customer_id#1, amount#2, status#3, region#4]

(4) HashAggregate
Input [2]: [amount#2, region#4]
Keys [1]: [region#4]
Functions [2]: [partial_sum(amount#2), partial_count(1)]
Aggregate Attributes [3]: [sum#17, isEmpty#18, count#19L]
Results [4]: [region#4, sum#20, isEmpty#21, count#22L]

In [7]:
# Compare two operations:
# One causes a shuffle, one does not

import time

# Operation 1: filter — NO shuffle (each partition works independently)
start = time.time()
result1 = df.filter(F.col("status") == "COMPLETED").count()
time1 = time.time() - start

# Operation 2: groupBy — CAUSES shuffle (data must move between partitions)
start = time.time()
result2 = df.groupBy("region").count().collect()
time2 = time.time() - start

print(f"Filter (no shuffle):    {time1:.3f} seconds | rows={result1}")
print(f"GroupBy (with shuffle): {time2:.3f} seconds | groups={len(result2)}")
print(f"Shuffle overhead:       {((time2-time1)/time1*100):.1f}% slower")

Filter (no shuffle):    1.050 seconds | rows=2477
GroupBy (with shuffle): 1.230 seconds | groups=4
Shuffle overhead:       17.2% slower


In [8]:
# Create a customers table to join with orders
customers_rows = []
for i in range(1, 5001):
    customers_rows.append((
        i,
        random.choice(["GOLD", "SILVER", "BRONZE"]),
        random.choice(["IN-SOUTH", "IN-NORTH", "IN-WEST", "IN-EAST"]),
    ))

customers_schema = StructType([
    StructField("customer_id", IntegerType(), False),
    StructField("tier",        StringType(),  True),
    StructField("home_region", StringType(),  True),
])

customers_df = spark.createDataFrame(customers_rows, customers_schema)

print(f"Orders rows:    {df.count():,}")
print(f"Customers rows: {customers_df.count():,}")

Orders rows:    10,000
Customers rows: 5,000


In [9]:
# JOIN 1 — Sort Merge Join (default, causes shuffle)
import time

start = time.time()
joined_df = df.join(customers_df, on="customer_id", how="inner")
result1 = joined_df.count()
time1 = time.time() - start

print(f"Sort Merge Join: {time1:.3f} seconds | rows={result1:,}")
print()

# See the plan — look for SortMergeJoin in the output
print("=== SORT MERGE JOIN PLAN ===")
joined_df.explain(mode="formatted")

Sort Merge Join: 1.844 seconds | rows=10,000

=== SORT MERGE JOIN PLAN ===
== Physical Plan ==
AdaptiveSparkPlan (9)
+- Project (8)
   +- SortMergeJoin Inner (7)
      :- Sort (3)
      :  +- Exchange (2)
      :     +- Scan ExistingRDD (1)
      +- Sort (6)
         +- Exchange (5)
            +- Scan ExistingRDD (4)


(1) Scan ExistingRDD
Output [5]: [order_id#0, customer_id#1, amount#2, status#3, region#4]
Arguments: [order_id#0, customer_id#1, amount#2, status#3, region#4], MapPartitionsRDD[4] at applySchemaToPythonRDD at NativeMethodAccessorImpl.java:0, ExistingRDD, UnknownPartitioning(0)

(2) Exchange
Input [5]: [order_id#0, customer_id#1, amount#2, status#3, region#4]
Arguments: hashpartitioning(customer_id#1, 8), ENSURE_REQUIREMENTS, [plan_id=432]

(3) Sort
Input [5]: [order_id#0, customer_id#1, amount#2, status#3, region#4]
Arguments: [customer_id#1 ASC NULLS FIRST], false, 0

(4) Scan ExistingRDD
Output [3]: [customer_id#50, tier#51, home_region#52]
Arguments: [customer_id#50

In [10]:
# JOIN 2 — Broadcast Join (no shuffle for small table)
from pyspark.sql.functions import broadcast

start = time.time()
broadcast_joined_df = df.join(
    broadcast(customers_df),
    on="customer_id",
    how="inner"
)
result2 = broadcast_joined_df.count()
time2 = time.time() - start

print(f"Sort Merge Join:  {time1:.3f} seconds | rows={result1:,}")
print(f"Broadcast Join:   {time2:.3f} seconds | rows={result2:,}")
print(f"Speedup:          {time1/time2:.1f}x faster")
print()
print("=== BROADCAST JOIN PLAN ===")
broadcast_joined_df.explain(mode="formatted")

Sort Merge Join:  1.844 seconds | rows=10,000
Broadcast Join:   1.502 seconds | rows=10,000
Speedup:          1.2x faster

=== BROADCAST JOIN PLAN ===
== Physical Plan ==
AdaptiveSparkPlan (6)
+- Project (5)
   +- BroadcastHashJoin Inner BuildRight (4)
      :- Scan ExistingRDD (1)
      +- BroadcastExchange (3)
         +- Scan ExistingRDD (2)


(1) Scan ExistingRDD
Output [5]: [order_id#0, customer_id#1, amount#2, status#3, region#4]
Arguments: [order_id#0, customer_id#1, amount#2, status#3, region#4], MapPartitionsRDD[4] at applySchemaToPythonRDD at NativeMethodAccessorImpl.java:0, ExistingRDD, UnknownPartitioning(0)

(2) Scan ExistingRDD
Output [3]: [customer_id#50, tier#51, home_region#52]
Arguments: [customer_id#50, tier#51, home_region#52], MapPartitionsRDD[28] at applySchemaToPythonRDD at NativeMethodAccessorImpl.java:0, ExistingRDD, UnknownPartitioning(0)

(3) BroadcastExchange
Input [3]: [customer_id#50, tier#51, home_region#52]
Arguments: HashedRelationBroadcastMode(List(cas